In [5]:
%run BitWise_To_ArithmeticLib.ipynb
%run Find_WitnessLib.ipynb

In [8]:
def fun_to_circuit_Arithmetic(C_BitW,word_size=64,is_input_a_list = False):
    if(word_size%2)!=0:
        print("word_size must be even")
        return 
    
    class ArithmeticCircuitExtended(ArithmeticCircuit):
        class Operations(ArithmeticCircuit.Operations):
            class LK(Operation.Variadic):
                T : Param.Tuple() = ()
                def eval(self,*operands):
                    for i in range(len(operands)):
                        if not operands[i] in self.T:
                            #print("v[%d] = %d"%(i,operands[i]))
                            #print("T =",self.T)
                            #print("NOT IN T")
                            return 1
                    return 0
                
                
            class LK_POW(Operation.Variadic):
                T : Param.Tuple() = ()
                def eval(self,*operands):
                    for i in range(len(operands)//2):
                        if not (operands[2*i],operands[2*i+1]) in self.T:
                            #print("v[%d] = (%d,%d)"%(i,operands[2*i],operands[2*i+1]))
                            #print("T =",self.T)
                            #print("NOT IN T")
                            return 1
                    return 0
            
            class AND(Operation.Variadic):
                def eval(self,*operands):
                    for i in range(len(operands)):
                        #print("b[%d] = %d"%(i,operands[i]))
                        if operands[i] != 0:
                            #print("b[%d] = %d"%(i,operands[i]))
                            return False
                    return True
    
    p = 52435875175126190479447740508185965837690552500527637822603658699938581184513
    
    C_Arith = ArithmeticCircuitExtended(base_ring=GF(p))    
    
    #cpt_x is the number of public inputs we already have met
    cpt_x = 0
    
    #cpt_w is the number of private inputs we already have met
    cpt_w = 0
    
    #var_calc is the list of the variables obtained by an add_input operations or a arithmetic computation
    var_calc = []
    
    #var_verif is the list of the variables obtained by a RangeProof operation
    var_verif = []
    
    dict_AND = {}
    dict_XOR = {}
    dict_OR = {}
    dict_NOT = {}
    dict_ADD = {}
    dict_SUB = {}
    dict_NEG = {}
    dict_MUL = {}
    dict_SHL = {}
    dict_SHR = {}
    
    #dict_decomp is a dictionnary containing at x the value x_e and x_o
    dict_decomp = {}
    
    #dict_decomp is a dictionnary containing at x the value x_e and x_o
    dict_cst = {}
    
    #dict_decomp is a dictionnary containing at (a,b) the value (q,r) (a=b*q+r) 
    #If neither the DIV or MOD has benn not proved, then there is not case (a,b)
    dict_DIV_MOD = {}
    
    dict_proof = {}
    
    #print("FRONTEND :")
    for node in C_BitW.nodes :
        
        #C_Arith.digraph().view()
        
        if node.is_INPUT():
        
            c,to_verif,cpt_x,cpt_w = inp_BitW_to_Arith(C_Arith,cpt_x,cpt_w,dict_decomp)
            
        elif node.is_CONST():
            
            cst = node.operation.value.integer_representation()
            c = add_cst_Arith(C_Arith,cst,word_size,dict_decomp,dict_cst)
            
        else : 

            a,b = get_inputs_node(node,var_calc)
            
            #print(a,b,end = ' : ')

            if node.is_AND():
                
                c,to_verif,cpt_w,need_proof = AND_BitW_to_Arith(C_Arith,a,b,cpt_w,dict_decomp,dict_AND)

            elif node.is_OR():
                
                c,to_verif,cpt_w,need_proof = OR_BitW_to_Arith(C_Arith,a,b,cpt_w,dict_decomp,dict_OR,dict_XOR,dict_AND)

            elif node.is_XOR():
                
                c,to_verif,cpt_w,need_proof = XOR_BitW_to_Arith(C_Arith,a,b,cpt_w,dict_decomp,dict_XOR)

            elif node.is_NOT():
                
                c,to_verif,cpt_w,need_proof = NOT_BitW_to_Arith(C_Arith,a,cpt_w,word_size,dict_decomp,dict_cst,dict_NOT,dict_XOR)
            
            elif node.is_DIV():
                
                c,to_verif,cpt_w,need_proof = DIV_BitW_to_Arith(C_Arith,a,b,cpt_w,dict_decomp,dict_DIV_MOD)

            elif node.is_MOD():

                c,to_verif,cpt_w,need_proof = MOD_BitW_to_Arith(C_Arith,a,b,cpt_w,dict_decomp,dict_DIV_MOD)

            elif node.is_ADD(): 

                c,to_verif,cpt_w,need_proof = ADD_BitW_to_Arith(C_Arith,a,b,cpt_w,word_size,dict_decomp,dict_ADD)
                
            elif node.is_SUB():

                c,to_verif,cpt_w,need_proof = SUB_BitW_to_Arith(C_Arith,a,b,cpt_w,word_size,dict_decomp,dict_SUB)

            elif node.is_MUL():

                c,to_verif,cpt_w,need_proof = MUL_BitW_to_Arith(C_Arith,a,b,cpt_w,word_size,dict_decomp,dict_MUL)

            elif node.is_NEG():

                c,to_verif,cpt_w,need_proof = NEG_BitW_to_Arith(C_Arith,a,cpt_w,word_size,dict_decomp,dict_cst,dict_NEG)

            elif node.is_SHL():

                b = node.operation.shift
                c,to_verif,cpt_w,need_proof = SHL_BitW_to_Arith(C_Arith,a,b,cpt_w,word_size,dict_decomp,dict_cst,dict_SHL,dict_MUL)

            elif node.is_SHR():

                b = node.operation.shift
                c,to_verif,cpt_w,need_proof = SHR_BitW_to_Arith(C_Arith,a,b,cpt_w,word_size,dict_decomp,dict_cst,dict_SHR,dict_DIV_MOD)

            #elif node.is_ROL():

            #elif node.is_ROR():
            
            dict_proof[node] = need_proof
            
        var_calc.append(c)  
        var_verif += to_verif
        if node.is_OUTPUT():
            C_Arith.add_output(c)
            

    list_of_nodes_to_LK = dict_of_list_to_list(dict_decomp)
    big_LK = C_Arith.LK(EvenBits(word_size//2))(*list_of_nodes_to_LK)
    
    tmp = var_verif+[big_LK]
    #for i in range(len(tmp)):
        #print("b[%d] ="%i,tmp[i])
    is_all_and = C_Arith.AND()(*(var_verif+[big_LK]))
    C_Arith.add_output(is_all_and)
    
    #C_Arith.digraph().view("Arithmetic_Circuit_for_%s_gate"%C_BitW.name)
    #C_BitW.digraph().view("BitWise_Circuit_for_%s_gate"%C_BitW.name)
    
    return C_Arith,dict_proof